# 📓 Day 1: Document Ingestion & Section-Aware Chunking
## VERA (Verified Evidence Retrieval Assistant)

**Objective**: Parse clinical guideline PDFs, preserve metadata (document name, page number, section), and generate structured chunks.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.ingestion.pdf_loader import PDFLoader
from src.ingestion.chunker import MedicalChunker
from src.utils.helpers import save_json
from tabulate import tabulate

print("Imports loaded successfully!")

Imports loaded successfully!


### 1. Load Clinical PDFs with Page & Table Extraction

In [2]:
pdf_dir = "../data/raw_pdfs"
loader = PDFLoader(extract_tables=True)
pages_data = loader.load_directory(pdf_dir)

print(f"Total pages extracted across all PDFs: {len(pages_data)}")
if pages_data:
    sample_page = pages_data[0]
    print(f"Sample Doc: {sample_page['doc_name']}, Page: {sample_page['page_number']}")
    print(f"Preview: {sample_page['text'][:250]}...")

2026-08-16 21:51:30 | INFO     | src.ingestion.pdf_loader:21 - Extracting PDF: ClinPediatr_2023_SMA_Treatment_Best_Practices.pdf
2026-08-16 21:51:32 | SUCCESS  | src.ingestion.pdf_loader:49 - Extracted 13 pages from ClinPediatr_2023_SMA_Treatment_Best_Practices.pdf
2026-08-16 21:51:32 | INFO     | src.ingestion.pdf_loader:21 - Extracting PDF: GenomeResearch_2024_LongRead_Chromosomal_Rearrangements.pdf
2026-08-16 21:51:34 | SUCCESS  | src.ingestion.pdf_loader:49 - Extracted 11 pages from GenomeResearch_2024_LongRead_Chromosomal_Rearrangements.pdf
2026-08-16 21:51:34 | INFO     | src.ingestion.pdf_loader:21 - Extracting PDF: IJMS_2023_SMA_Past_Present_Future_Review.pdf
2026-08-16 21:51:40 | SUCCESS  | src.ingestion.pdf_loader:49 - Extracted 38 pages from IJMS_2023_SMA_Past_Present_Future_Review.pdf
2026-08-16 21:51:40 | INFO     | src.ingestion.pdf_loader:21 - Extracting PDF: medRxiv_2024_SMA_Missed_Diagnoses_Sequencing.pdf
2026-08-16 21:51:42 | SUCCESS  | src.ingestion.pdf_loader:49 - E

### 2. Perform Section-Aware Chunking

In [3]:
chunker = MedicalChunker(chunk_size=500, chunk_overlap=80, min_chunk_length=40)
chunks = chunker.chunk_pages(pages_data)

print(f"Total structured chunks created: {len(chunks)}")

# Display sample chunk table
table_data = []
for c in chunks[:5]:
    table_data.append([
        c.chunk_id,
        c.doc_name[:30] + "...",
        c.section[:25],
        c.page_number,
        c.token_count,
        c.content[:60] + "..."
    ])

print(tabulate(table_data, headers=["Chunk ID", "Document", "Section", "Page", "Words", "Snippet"], tablefmt="grid"))

2026-08-16 21:51:42 | SUCCESS  | src.ingestion.chunker:114 - Generated 94 structured chunks across 83 pages.
Total structured chunks created: 94
+------------+-----------------------------------+------------------+--------+---------+-----------------------------------------------------------------+
| Chunk ID   | Document                          | Section          |   Page |   Words | Snippet                                                         |
+============+===================================+==================+========+=========+=================================================================+
| fb4d64b2   | ClinPediatr_2023_SMA_Treatment... | General Overview |      1 |     158 | RESEARCHARTICLE OPENACCESS Spinal Muscular Atrophy Update in... |
+------------+-----------------------------------+------------------+--------+---------+-----------------------------------------------------------------+
| 9c436f60   | ClinPediatr_2023_SMA_Treatment... | General Overview |      2 |  

### 3. Save Structured Chunks Catalog for Vector Store

In [4]:
chunks_dict = [c.model_dump() for c in chunks]
save_json(chunks_dict, "../data/processed/chunk_catalog.json")
print("Successfully saved chunks to '../data/processed/chunk_catalog.json'!")

Successfully saved chunks to '../data/processed/chunk_catalog.json'!
